# Конспект 4. Работа с файлами

До сих пор данные жили в ячейках ноутбука и пропадали, как только закрывался Jupyter.
Файлы же хранятся на диске ПК, т.е. записали один раз и прочитали когда угодно, хоть другой
программой.

Практическая любая работа с файлами выполняется по следующей схеме:

**создать файл -> записать -> открыть -> прочитать -> обработать**
Читайте и **запускайте каждую ячейку по порядку**,
некоторые примеры используют файлы, которые создали предыдущие.

## Данные конспекта

In [1]:
TEXTS = [
    "Учёные открыли новый вид бактерий в глубинах Тихого океана",
    "Сборная выиграла матч со счётом 3:1",
    "Новый вирус обнаружен учёными в лаборатории Новосибирска",
]

LABELS = ["наука", "спорт", "наука"]
SCORES = [5, 3, 4]

print("записей:", len(TEXTS))

записей: 3


---

# Часть I. Основы

# 1. Открыть, записать, прочитать

Файл открывает функция `open(имя, режим, encoding='utf-8')`.

| Режим | Что делает | Если файла нет | Если файл есть |
| --- | --- | --- | --- |
| `'r'` | чтение (по умолчанию) | ошибка `FileNotFoundError` | читаем с начала |
| `'w'` | запись | создаёт | **стирает** и пишет заново |
| `'a'` | дописать в конец | создаёт | дописывает, старое не трогает |

`encoding='utf-8'` указывайте **всегда**, и при записи, и при чтении. Без него русский текст может отображаться странными символами `РџСЂРёРІРµС‚`.

- `file.write(строка)` записывает строку. Перенос строки `\n` нужно ставить **самому**.
- `file.read()` возвращает всё содержимое файла одной строкой.
- `file.close()` закрывает файл. Пока файл не закрыт, записанное содержимое может ещё не записаться на диск.

In [2]:
file = open('first.txt', 'w', encoding='utf-8')     # создать и открыть на запись
file.write(TEXTS[0] + '\n')
file.write(LABELS[0] + '\n')
file.close()

file = open('first.txt', 'r', encoding='utf-8')     # открыть на чтение
text = file.read()
file.close()

print(text)
print('символов в файле:', len(text))

Учёные открыли новый вид бактерий в глубинах Тихого океана
наука

символов в файле: 65


Файл `first.txt` появился в папке рядом с ноутбуком. Откройте его двойным щелчком
в списке файлов Jupyter и убедитесь, что внутри хранятся две строки.

---

# 2. `with` - файл закроется сам

Забыть `close()` легко, поэтому файлы открывают через `with`. Весь код в блоке `with`, работает с открытым файлом. Как только блок закончился, файл закрывается, 
даже если внутри случилась ошибка.

Старайтесь работать именно так в практике и будущем.

In [3]:
with open('first.txt', 'r', encoding='utf-8') as file:
    text = file.read()

print(text)
print('файл закрыт?', file.closed)

Учёные открыли новый вид бактерий в глубинах Тихого океана
наука

файл закрыт? True


---

# 3. Список в файл, файл по строкам

Список обычно записывают при помощи **цикла**, где каждый элемент - это отдельная строка, с `\n` в конце.

Читают такие файл тоже циклом, `for line in file` выдаёт строки файла по одной. В конце каждой
строки остаётся `\n`, поэтому её чистят методом `strip()`. Номера строк можно учитывать при помощи `enumerate`.

In [4]:
with open('texts.txt', 'w', encoding='utf-8') as file:
    for text in TEXTS:
        file.write(text + '\n')

with open('texts.txt', 'r', encoding='utf-8') as file:
    for number, line in enumerate(file, start=1):
        print(f'{number}. {line.strip()}')

1. Учёные открыли новый вид бактерий в глубинах Тихого океана
2. Сборная выиграла матч со счётом 3:1
3. Новый вирус обнаружен учёными в лаборатории Новосибирска


Что будет без `strip()`? Сам `\n` при печати не виден, но между строками появятся пустые:
один перенос из файла и второй от `print`. Увидеть его можно через `repr`:

In [5]:
with open('texts.txt', 'r', encoding='utf-8') as file:
    first_line = file.readline()       # одна строка

print(repr(first_line))

'Учёные открыли новый вид бактерий в глубинах Тихого океана\n'


---

# 4. Числа в файле - это ТОЖЕ строки

`write()` принимает **только строки**, `file.write(5)` выведет ошибку `TypeError`.
Для обрабоки чисел их сначала превращают в строку, как `str(5)` или f-строкой `f'{score}'` после чего записывают.
Возвращает файл точно также только строки, так `'5' + '3'` - это `'53'`,
поэтому прочитанное число изначально преобразуют в число при помощи `int`.

In [6]:
with open('scores.txt', 'w', encoding='utf-8') as file:
    for score in SCORES:
        file.write(str(score) + '\n')

scores = []
with open('scores.txt', 'r', encoding='utf-8') as file:
    for line in file:
        scores.append(int(line))         # int() сам справляется с '\n' на конце

print(scores, 'сумма:', sum(scores))

[5, 3, 4] сумма: 12


---

# 5. Добавить строки в конец файла (`'a'`)

Режим `'a'` (append) дописывает строки в конец и ничего не стирает. Так обычно записывают данные в журналы или при логировании, здесь
каждая новая запись записывается в конец файла.

`readlines()` читает **все** строки сразу и возвращает список. С ним работают
как с любым списком: `len()`, срезы `[-2:]`, цикл.

> Если запустить ячейку с `'a'` дважды, строки допишутся дважды.

In [7]:
with open('texts.txt', 'a', encoding='utf-8') as file:
    file.write('Учёные нашли воду на Марсе\n')

with open('texts.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()

print('строк:', len(lines))
print(lines[-2:])                        # две последние, видно \n на конце

строк: 4
['Новый вирус обнаружен учёными в лаборатории Новосибирска\n', 'Учёные нашли воду на Марсе\n']


Открытый файл читается **один раз**, как генератор из темы 3: после `read()` или
`readlines()` второй вызов вернёт пустоту. Нужно прочитать снова - откройте файл заново.

---

# 6. Перевернуть и склеить

**Перевернуть.** Читаем строки списком, переворачиваем срезом `[::-1]`, записываем в
**новый** файл. Строки из `readlines()` уже с символом `\n`, поэтому их записывает
`writelines()` как есть.

In [8]:
with open('texts.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()

with open('texts_reversed.txt', 'w', encoding='utf-8') as file:
    file.writelines(lines[::-1])

with open('texts_reversed.txt', 'r', encoding='utf-8') as file:
    print(file.read())

Учёные нашли воду на Марсе
Новый вирус обнаружен учёными в лаборатории Новосибирска
Сборная выиграла матч со счётом 3:1
Учёные открыли новый вид бактерий в глубинах Тихого океана



**Склеить.** Имена файлов храним в списке и обходим циклом. Имя удобно собрать
f-строкой. Тексты складываем в список и склеиваем `join`: разделитель встаёт
**между** частями, а не после последней.

In [10]:
names = []
for i in range(len(TEXTS)):
    name = f'text_{i + 1}.txt'           # text_1.txt, text_2.txt, ...
    names.append(name)
    with open(name, 'w', encoding='utf-8') as file:
        file.write(TEXTS[i] + '\n')

parts = []
for name in names:
    with open(name, 'r', encoding='utf-8') as file:
        parts.append(file.read())

with open('all_texts.txt', 'w', encoding='utf-8') as file:
    file.write('---\n'.join(parts))

with open('all_texts.txt', 'r', encoding='utf-8') as file:
    print(file.read())

Учёные открыли новый вид бактерий в глубинах Тихого океана
---
Сборная выиграла матч со счётом 3:1
---
Новый вирус обнаружен учёными в лаборатории Новосибирска



---

# Часть II. Обработка данных

# 7. Слова и их частота из файла

Прочитанный текст, явялется обычной строкой, с ней работают все примеры из **темы 1**. Режем на слова, приводим к нижнему регистру и убираем знаки препинания по краям (`strip('.,:')`), 
иначе `матч` и `матч.` будут разными словами.

Частоту слов можно вычислять при помощи словарей и сортировать по убыванию значения (`key=` из **темы 2**).
Короткие служебные слова (`в`, `со`) обычно не учитывают и отсекают по длине слов.

In [11]:
with open('texts.txt', 'r', encoding='utf-8') as file:
    text = file.read()

words = [word.strip('.,:').lower() for word in text.split()]
print('всего слов:', len(words))
print('разных слов:', len(set(words)))

freq = {}
for word in words:
    if len(word) >= 4:
        freq[word] = freq.get(word, 0) + 1

top = sorted(freq.items(), key=lambda pair: pair[1], reverse=True)
print('чаще всего от 4 букв:', top[:3])

всего слов: 27
разных слов: 24
чаще всего от 4 букв: [('учёные', 2), ('новый', 2), ('открыли', 1)]


---

# 8. Отбор строк в новый файл

Читаем строки, проверяем каждую и в новый файл пишем только подходящие. Если в
строке несколько полей через разделитель, например `наука | Учёные открыли...`,
их можно разделить при помощи метода `split(' | ')`, где `[0]` - это первое поле, `[1]` - второе.

- `line.startswith('ERROR')` - строка начинается с `ERROR`;
- `'ERROR' in line` - `ERROR` есть где-то в строке.

In [12]:
with open('labeled.txt', 'w', encoding='utf-8') as file:
    for i in range(len(TEXTS)):
        file.write(f'{LABELS[i]} | {TEXTS[i]}\n')

with open('labeled.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()

count = 0
with open('science.txt', 'w', encoding='utf-8') as file:
    for line in lines:
        label = line.split(' | ')[0]
        if label == 'наука':
            file.write(line)             # строка уже с \n
            count += 1

with open('science.txt', 'r', encoding='utf-8') as file:
    print(file.read())
print('отобрано:', count)

наука | Учёные открыли новый вид бактерий в глубинах Тихого океана
наука | Новый вирус обнаружен учёными в лаборатории Новосибирска

отобрано: 2


---

# 9. CSV: таблица в текстовом файле

**CSV** - это таблица, записанная текстом. Здесь первая строка - это заголовок, а оастальное значения разделены
запятыми. Его можно открыть и Excel-ем, и pandas, и любым текстовый редактором.

```
label,score
наука,5
спорт,3
```

Читаем так: первую строку пропускаем через `readline()`, остальные разделяем при помощи метода `split(',')` и превращаем числа в `int`. Итоги по группам накапливаем в словаре, после чего записываем в новый CSV.

In [13]:
# создаём CSV
with open('scores.csv', 'w', encoding='utf-8') as file:
    file.write('label,score\n')
    for i in range(len(LABELS)):
        file.write(f'{LABELS[i]},{SCORES[i]}\n')

# читаем и считаем итоги по меткам
counts = {}
sums = {}
with open('scores.csv', 'r', encoding='utf-8') as file:
    file.readline()                              # пропускаем заголовок
    for line in file:
        label, score = line.strip().split(',')
        counts[label] = counts.get(label, 0) + 1
        sums[label] = sums.get(label, 0) + int(score)

# записываем итоговый CSV
with open('summary.csv', 'w', encoding='utf-8') as file:
    file.write('label,texts,total_score\n')
    for label in counts:
        file.write(f'{label},{counts[label]},{sums[label]}\n')

with open('summary.csv', 'r', encoding='utf-8') as file:
    print(file.read())

label,texts,total_score
наука,2,9
спорт,1,3



> `split(',')` ломается, если запятая есть внутри значения (`"New York, NY"`).

---

## Шпаргалка

| Что нужно | Как пишется | Важно |
| --- | --- | --- |
| создать / перезаписать | `with open('f.txt', 'w', encoding='utf-8') as file:` | старое **стирается** |
| дописать в конец | `with open('f.txt', 'a', encoding='utf-8') as file:` | повторный запуск допишет ещё |
| прочитать | `with open('f.txt', 'r', encoding='utf-8') as file:` | нет файла - `FileNotFoundError` |
| записать строку | `file.write(text + '\n')` | `\n` сам не ставится |
| записать число | `file.write(str(n) + '\n')` | только строки |
| прочитать всё | `text = file.read()` | одна строка |
| список строк | `lines = file.readlines()` | в строках `\n` |
| перебрать строки | `for line in file:` + `line.strip()` | |
| с номерами | `for i, line in enumerate(file, start=1):` | |
| число из файла | `int(line)` | из файла приходят строки |
| перевернуть | `file.writelines(lines[::-1])` | в новый файл |
| склеить | `'---\n'.join(parts)` | разделитель между частями |
| поле строки | `line.split(' | ')[0]` | |
| строка CSV | `a, b = line.strip().split(',')` | |
| пропустить заголовок | `file.readline()` перед циклом | |
| второй раз | открыть файл заново | прочитанный файл пуст |

---

# Найди ошибку

Раскомментируйте, запустите и найдите причину. Ответы - после ячеек.

In [ ]:
# Ошибка 1 - ошибки нет, но в файле одна строка вместо трёх
# for text in TEXTS:
#     with open('log.txt', 'w', encoding='utf-8') as file:
#         file.write(text + '\n')
#
# with open('log.txt', 'r', encoding='utf-8') as file:
#     print(file.readlines())

In [ ]:
# Ошибка 2
# with open('scores.txt', 'w', encoding='utf-8') as file:
#     for score in SCORES:
#         file.write(score)

In [ ]:
# Ошибка 3 - ошибки нет, но всё в одну строку
# with open('labels.txt', 'w', encoding='utf-8') as file:
#     for label in LABELS:
#         file.write(label)
#
# with open('labels.txt', 'r', encoding='utf-8') as file:
#     print(file.read())

In [ ]:
# Ошибка 4 - ошибки нет, но слов 0
# with open('texts.txt', 'r', encoding='utf-8') as file:
#     lines = file.readlines()
#     text = file.read()
#
# print('строк:', len(lines), 'слов:', len(text.split()))

In [ ]:
# Ошибка 5
# total = 0
# with open('scores.csv', 'r', encoding='utf-8') as file:
#     for line in file:
#         label, score = line.strip().split(',')
#         total += int(score)
#
# print(total)

**Ответы:**

1. Файл открывается в режиме `'w'` **внутри** цикла и каждый раз стирается. Остаётся
   только последняя запись. Нужно открыть файл один раз, до цикла, а цикл поставить
   под `with`.
2. `TypeError: write() argument must be str, not int`. Нужно `file.write(str(score) + '\n')`.
3. `write` не ставит перенос строки: в файле `наукаспортнаука`. Нужно `file.write(label + '\n')`.
4. `readlines()` уже прочитал файл до конца, `read()` вернул пустую строку. Нужно
   что-то одно: `text = ''.join(lines)` или открыть файл заново.
5. `ValueError: invalid literal for int() with base 10: 'score'`. Первая строка CSV -
   заголовок, и `int('score')` не работает. Перед циклом нужно `file.readline()`.